# Experiment Tracking in Snowflake

Extracted from [**Distributed Hyperparameter Tuning with Experiment Tracking in Snowflake**](https://www.snowflake.com/en/developers/guides/hpo-with-experiment-tracking/).

We'll build a classification model using the Wine Quality dataset.

In [ ]:
%%sql -r dataframe_1
USE test.public;

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from xgboost import XGBClassifier

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import Session
from snowflake.ml.experiment.experiment_tracking import ExperimentTracking
from snowflake.ml.modeling import tune
from snowflake.ml.modeling.tune.search import RandomSearch, BayesOpt
from snowflake.ml.data.data_connector import DataConnector
from snowflake.ml.runtime_cluster import scale_cluster

# Get active Snowflake session
session = get_active_session()
print(f"Connected to Snowflake: {session.get_current_database()}.{session.get_current_schema()}")

# Create dated experiment name for tracking runs over time
experiment_date = datetime.now().strftime("%Y%m%d")
experiment_name = f"Wine_Quality_Classification_{experiment_date}"
print(f"\nExperiment Name: {experiment_name}")


### Generate Wine Quality Classification Dataset

We'll create a synthetic dataset inspired by wine quality prediction. The goal is to classify wines as high quality (1) or standard quality (0) based on chemical properties.

In [ ]:
# Generate synthetic wine quality dataset
np.random.seed(42)
n_samples = 20000

# Feature generation with realistic correlations
data = {
    "FIXED_ACIDITY": np.random.normal(7.0, 1.5, n_samples),
    "VOLATILE_ACIDITY": np.random.gamma(2, 0.2, n_samples),
    "CITRIC_ACID": np.random.beta(2, 5, n_samples),
    "RESIDUAL_SUGAR": np.random.lognormal(1, 0.8, n_samples),
    "CHLORIDES": np.random.gamma(3, 0.02, n_samples),
    "FREE_SULFUR_DIOXIDE": np.random.normal(30, 15, n_samples),
    "TOTAL_SULFUR_DIOXIDE": np.random.normal(120, 40, n_samples),
    "DENSITY": np.random.normal(0.997, 0.003, n_samples),
    "PH": np.random.normal(3.2, 0.3, n_samples),
    "SULPHATES": np.random.gamma(4, 0.15, n_samples),
    "ALCOHOL": np.random.normal(10.5, 1.5, n_samples)
}

df = pd.DataFrame(data)

# Create quality target based on feature combinations
quality_score = (
    0.3 * (df["ALCOHOL"] - df["ALCOHOL"].mean()) / df["ALCOHOL"].std() +
    0.2 * (df["CITRIC_ACID"] - df["CITRIC_ACID"].mean()) / df["CITRIC_ACID"].std() -
    0.25 * (df["VOLATILE_ACIDITY"] - df["VOLATILE_ACIDITY"].mean()) / df["VOLATILE_ACIDITY"].std() +
    0.15 * (df["SULPHATES"] - df["SULPHATES"].mean()) / df["SULPHATES"].std() +
    np.random.normal(0, 0.3, n_samples)  # Add noise
)

# Binary classification: 1 = high quality, 0 = standard quality
df["QUALITY"] = (quality_score > quality_score.quantile(0.6)).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:\n{df['QUALITY'].value_counts()}")
print(f"\nFeature statistics:\n{df.describe()}")


### Prepare Train/Validation/Test Splits

In [ ]:
# Separate features and target
X = df.drop('QUALITY', axis=1)
y = df['QUALITY']

# Create train/val/test splits
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp)

# Scale features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Validation set: {X_val_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")


## Baseline Model with Experiment Tracking

In [ ]:
# Initialize Experiment Tracking
exp = ExperimentTracking(session=session)
exp.set_experiment(experiment_name)

# Note: Snowflake supports autologging for certain ML frameworks, but this example uses 
# explicit logging (exp.log_params, exp.log_metrics) to demonstrate a framework-agnostic 
# approach. Explicit logging works with any ML library (scikit-learn, XGBoost, PyTorch, 
# TensorFlow, custom frameworks, etc.) and gives you precise control over what gets logged, 
# without requiring integration with Snowflake's modeling APIs.

# Train baseline model
with exp.start_run(run_name="baseline_xgboost") as run:
    # Define baseline parameters
    baseline_params = {
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0.1,
        'min_child_weight': 8,
        'random_state': 42,
    }
    
    # Log parameters
    exp.log_params(baseline_params)
    
    # Train model
    baseline_model = XGBClassifier(**baseline_params)
    baseline_model.fit(X_train_scaled, y_train)
    
    # Evaluate on validation set
    y_val_pred = baseline_model.predict(X_val_scaled)
    y_val_proba = baseline_model.predict_proba(X_val_scaled)[:, 1]
    
    # Calculate metrics
    val_metrics = {
        'val_accuracy': metrics.accuracy_score(y_val, y_val_pred),
        'val_precision': metrics.precision_score(y_val, y_val_pred),
        'val_recall': metrics.recall_score(y_val, y_val_pred),
        'val_f1': metrics.f1_score(y_val, y_val_pred),
        'val_roc_auc': metrics.roc_auc_score(y_val, y_val_proba)
    }
    
    # Log metrics
    exp.log_metrics(val_metrics)
    
    print("Baseline Model Performance:")
    for metric, value in val_metrics.items():
        print(f"  {metric}: {value:.4f}")


## View Results in Snowflake UI

All experiment runs are now available in the Snowflake UI:

1. Navigate to **AI & ML > Experiments** in the left sidebar
2. Find the `Wine_Quality_Classification_YYYYMMDD` experiment (with today's date)
3. Compare runs, view metrics, and analyze results

**Note**: Each time you run this notebook on a different day, it creates a new dated experiment, allowing you to track model performance over time and across different data versions.

The UI provides:
- Side-by-side run comparisons
- Metric visualizations
- Parameter distributions
- Model artifacts and metadata
